# Voice control test: what Gemma 4 hears and which tool it calls

Type or say a sentence and see two lines: `you:` the words Gemma 4 E4B heard (the transcript), and `->` the tool it calls with the arguments it took from your words, for example `pick_place(object: small red cube, place: box)`. Both lines come from one answer of Gemma. No program works out a command or an argument from your words, and "stop robot" is a command of Gemma like every other.

**Kernel:** "SO-101 SmolVLA notebooks (voice workshop environment)". `./run_notebooks.sh` in this directory registers it and starts JupyterLab here; `./run_notebooks.sh --kernel-only` only registers it. In VS Code or Cursor: "Select Kernel" -> "Jupyter Kernel..." -> that name. The first cell stops with a message if another kernel is in use.

**What runs:** the Gemma client in `../voice`, a copy of the branch `gemma4-e4b-it_voice_intent` with the two stack commands and the `place` argument added (instructions in `vla_pipeline/llm/intents.py`, grammar in `config/intent.gbnf`). The model files, `llama-server` (the program of llama.cpp that runs the model) and the Python environment are those of `workshops/vvla-pipeline`, which has to be set up on this machine. Nothing moves a robot: the notebook shows the call and stops there.

| Section | What it does | Needs you to speak |
|---|---|---|
| 1 | Table of sentences you can say and the tool each one should call | no |
| 2 | Starts Gemma | no |
| 3 | One typed sentence | no |
| 4 | One sentence in a synthetic voice (espeak-ng, a speech synthesiser) | no |
| 5 | The whole table, typed and in the synthetic voice, with what Gemma called | no |
| 6 | Your voice, one recorded clip | yes |
| 7 | The branch's live voice session on your microphone | yes |
| 8 | Stops Gemma | no |

In [1]:
import atexit
import os
import sys
import time

RIGHT_KERNEL = ('Select the kernel "SO-101 SmolVLA notebooks (voice workshop environment)"; '
                'run_notebooks.sh in this directory registers it.')
VOICE_DIR = os.path.abspath(os.path.join('..', 'voice'))  # the Gemma client and this notebook's helpers
if not os.path.isfile(os.path.join(VOICE_DIR, 'notebook_tools.py')):
    raise RuntimeError('No notebook_tools.py in %s: the notebook has to run in its own directory.' % VOICE_DIR)
if VOICE_DIR not in sys.path:
    sys.path.insert(0, VOICE_DIR)
try:
    import sounddevice as sd
    from IPython.display import Audio, Markdown, display
    import notebook_tools as t  # server start, synthetic voice, example sentences; refuses a client that is not ../voice
    from vla_pipeline.llm.intents import command_from_json
    from vla_pipeline.llm.voice import _wav, open_voice
except ImportError as e:
    raise RuntimeError('%s. %s' % (e, RIGHT_KERNEL)) from e
if os.path.realpath(sys.prefix) != os.path.realpath(os.path.join(t.VOICE_WORKSHOP, '.venv')):
    raise RuntimeError('This kernel runs %s, not %s/.venv. %s' % (sys.prefix, t.VOICE_WORKSHOP, RIGHT_KERNEL))

# The kernel of this directory starts through the voice workshop's environment scripts (run_notebooks.sh).
# HSA_OVERRIDE_GFX_VERSION is the variable of scripts/ryzen_ai_env.sh that llama-server needs.
if not os.environ.get('HSA_OVERRIDE_GFX_VERSION'):
    raise RuntimeError('This kernel was started without the voice workshop environment. ' + RIGHT_KERNEL)
print('HSA_OVERRIDE_GFX_VERSION =', os.environ['HSA_OVERRIDE_GFX_VERSION'])

cfg = t.load_voice_config()
RATE, WINDOW_S = int(cfg['audio']['sample_rate']), float(cfg[t.SECTION]['window_s'])
# The microphone of sections 6 and 7. None is the system's default input; a number from `python -m sounddevice`
# names another one.
MIC = cfg['audio']['mic_device']
LOG_DIR = os.path.abspath('logs')  # the server's log; the directory is not in git
if 'espeak' not in globals():
    espeak, espeak_rate = t.open_espeak()

# The sentences and what each must give are not typed here: the sentences of the tool-call test (with a place)
# and the Gemma branch's own list. The branch's list names no place, so there the place is not checked (None).
EXAMPLES = [dict(say=s, command=c, object=o, place=p, source='tool-call test') for _, s, c, o, p in t.CASES]
EXAMPLES += [dict(say=s, command=i.value, object=o or '', place=None if i.value == 'pick_place' else '', source='branch')
             for s, i, o in t.llama_intent._SELFTEST_CASES if s not in {e['say'] for e in EXAMPLES}]
EXAMPLES.sort(key=lambda e: (e['command'] == 'unknown', e['command'] != 'pick_place'))  # with arguments, without, no command


def call_text(command, obj='', place=''):
    """A call as the spec writes it: pick_place(object: small red cube, place: box)."""
    if command == 'unknown':
        return 'no command'
    args = ', '.join('%s: %s' % (k, v) for k, v in (('object', obj), ('place', place)) if v)
    return '%s(%s)' % (command, args)


def expected_text(example):
    return call_text(example['command'], example['object'], example['place'] or '')


def answer(cmd, seconds=None):
    """One answer of Gemma (the client's ParsedCommand) as plain fields."""
    return dict(command=cmd.intent.value, object=cmd.params.get('object', ''), place=cmd.params.get('place', ''),
                heard=cmd.transcript, seconds=seconds)


def right(example, got):
    """The command; for pick_place also the object, and the place where the example names one."""
    if got['command'] != example['command']:
        return False
    if example['command'] != 'pick_place':
        return True
    return t.same(got['object'], example['object']) and (example['place'] is None or t.same(got['place'], example['place']))


def show(got, example=None):
    took = '' if got['seconds'] is None else '   (%.2f s)' % got['seconds']
    print('you:', got['heard'])
    print(' -> ', call_text(got['command'], got['object'], got['place']) + took)
    if example is not None:
        print('     expected', expected_text(example) + ':', 'right' if right(example, got) else 'WRONG')


def server():
    if globals().get('gemma') is None:
        raise RuntimeError('Gemma is not running: run the cell of section 2 first.')
    return gemma


def ask(text):
    """Typed: the client's parse()."""
    began = time.perf_counter()
    cmd = server().parse(text)
    return answer(cmd, time.perf_counter() - began)


def ask_clip(wav):
    """Spoken: one clip (the bytes of a WAV file) through the client's parse_audio()."""
    began = time.perf_counter()
    cmd = server().parse_audio(wav)
    return answer(cmd, time.perf_counter() - began)


def synthetic(text):
    """The sentence in the espeak-ng voice, inside a quiet clip as long as the voice session's window."""
    return t.clip(t.speak(espeak, espeak_rate, text, RATE), RATE, WINDOW_S)


print('%d example sentences; spoken clips are %.0f s at %d Hz' % (len(EXAMPLES), WINDOW_S, RATE))

HSA_OVERRIDE_GFX_VERSION = 11.0.0
30 example sentences; spoken clips are 6 s at 16000 Hz


## 1. What you can say

Each row is a sentence and the tool Gemma should call for it, with the arguments it should take from the sentence. Only `pick_place` has arguments: `object` (up to three words; colour and size are words of the object) and `place` (up to two words). `-` means the argument stays empty, `any` means the list the sentence comes from names no place, so the place is not checked. "none" in the tool column means the sentence must not start anything.

The column "List" says where a row comes from: `tool-call test` = the sentences of the tool-call test of SPEC.md section 4.5 (`CASES` in `../voice/notebook_tools.py`), `branch` = the Gemma branch's own list (`_SELFTEST_CASES` in `llama_intent.py`). You are not bound to these sentences: say it in your own words and see what Gemma does.

In [2]:
def shown(value):
    return 'any' if value is None else (value or '-')


lines = ['| # | Say | Tool | object | place | List |', '|---|---|---|---|---|---|']
lines += ['| %d | %s | %s | %s | %s | %s |' % (n, e['say'], 'none' if e['command'] == 'unknown' else '`%s`' % e['command'],
                                             shown(e['object']), shown(e['place']), e['source'])
          for n, e in enumerate(EXAMPLES, 1)]
display(Markdown('\n'.join(lines)))

| # | Say | Tool | object | place | List |
|---|---|---|---|---|---|
| 1 | pick up the small red cube and put it in the box | `pick_place` | small red cube | box | tool-call test |
| 2 | put the red cube in the box | `pick_place` | red cube | box | tool-call test |
| 3 | put the blue ball in the box | `pick_place` | blue ball | box | tool-call test |
| 4 | grab the large green cylinder and place it in the box | `pick_place` | large green cylinder | box | tool-call test |
| 5 | pick up the yellow cylinder | `pick_place` | yellow cylinder | - | tool-call test |
| 6 | take the small blue cube to the bin | `pick_place` | small blue cube | bin | tool-call test |
| 7 | place the large red ball in the bin | `pick_place` | large red ball | bin | tool-call test |
| 8 | pick up the ball | `pick_place` | ball | - | tool-call test |
| 9 | grab the bottle and hand it to me | `pick_place` | bottle | any | branch |
| 10 | learn the stack | `learn_stack` | - | - | tool-call test |
| 11 | rebuild the stack | `rebuild_stack` | - | - | tool-call test |
| 12 | stop robot | `stop` | - | - | tool-call test |
| 13 | go home | `home` | - | - | tool-call test |
| 14 | wave hello | `wave` | - | - | tool-call test |
| 15 | fetch the cube | `fetch_block` | - | - | tool-call test |
| 16 | show me your best dance moves | `dance` | - | - | tool-call test |
| 17 | can you copy my arm movements | `gesture_mimic` | - | - | branch |
| 18 | mirror my hand please | `gesture_mimic` | - | - | branch |
| 19 | go grab the block and bring it to me | `fetch_block` | - | - | branch |
| 20 | hold this pencil for me | `grip` | - | - | branch |
| 21 | release | `release` | - | - | branch |
| 22 | open your hand | `release` | - | - | branch |
| 23 | drop it | `release` | - | - | branch |
| 24 | do not rebuild the stack | none | - | - | tool-call test |
| 25 | the stack looks nice | none | - | - | tool-call test |
| 26 | I learned a lot today | none | - | - | tool-call test |
| 27 | who built this stack | none | - | - | tool-call test |
| 28 | the red cube is in the box | none | - | - | tool-call test |
| 29 | is the ball in the box | none | - | - | tool-call test |
| 30 | what's the weather like today | none | - | - | branch |

## 2. Start Gemma

Starts `llama-server` with Gemma 4 E4B through the branch's own client, with two request slots and a context of 4,096 tokens (SPEC.md section 5.1, departure 1). The cell prints every setting that differs from the branch's `config/pipeline.yaml`. The server's log is `logs/llama-server.log` in this directory. If something already answers on the port, the cell stops with a message instead of using a server whose settings it does not know.

In [3]:
if globals().get('gemma') is None:
    gemma, base, section = t.start_server(LOG_DIR, {'parallel': 2, 'ctx_size': 4096}, [])
    atexit.register(gemma.close)  # the server ends with the kernel if section 8 is not run
else:
    print('Gemma already runs for this notebook at', base)

Gemma branch commit 1d231e1 with this workshop's changes, client ../voice, section gemma4-e4b-it


settings used: {'port': 8082, 'n_gpu_layers': 99, 'ctx_size': 4096, 'parallel': 2, 'temperature': 0.0, 'no_warmup': True, 'hsa_override_gfx_version': '11.5.0', 'extra_args': None}


differences from config/pipeline.yaml (setting: file value -> used value): {'parallel': (1, 2), 'ctx_size': (2048, 4096)}


HSA_OVERRIDE_GFX_VERSION in the calling environment (wins over the setting if set): 11.0.0


server ready after 2.2 s, log logs/llama-server.log


## 3. Type a sentence

Change the sentence and run the cell again.

In [4]:
show(ask('pick up the small red cube and put it in the box'))

you: pick up the small red cube and put it in the box
 ->  pick_place(object: small red cube, place: box)   (1.55 s)


## 4. A sentence in the synthetic voice

The sentence is spoken by espeak-ng into a quiet clip as long as the voice session's window, and the clip is sent to Gemma the way the session sends one. `you:` is what Gemma heard in the sound.

In [5]:
show(ask_clip(synthetic('put the blue ball in the box')))

you: put the blue ball in the box
 ->  pick_place(object: blue ball, place: box)   (1.12 s)


## 5. The whole table, typed and in the synthetic voice

Every sentence of the table is sent once typed and once spoken. The table below grows while the cell runs: a sentence's row appears as soon as its two answers are in. An answer that differs from the expected call is marked **WRONG**: the command is compared, and for `pick_place` the object and (where the table names one) the place. The transcript is shown, not scored.

In [6]:
def called(example, got):
    text = '`%s`' % call_text(got['command'], got['object'], got['place'])
    return text if right(example, got) else text + ' **WRONG**'


def table(results):
    """The answers so far as a table, one row per sentence."""
    lines = ['| # | Say | Gemma should call | Typed: Gemma called | Spoken: Gemma heard | Spoken: Gemma called |',
             '|---|---|---|---|---|---|']
    lines += ['| %d | %s | `%s` | %s | %s | %s |' % (n, e['say'], expected_text(e), called(e, typed), spoken['heard'], called(e, spoken))
              for n, (e, typed, spoken) in enumerate(results, 1)]
    return Markdown('\n'.join(lines))


results = []  # (sentence, typed answer, spoken answer); the table and the two counts below come from this list
shown = display(table(results), display_id=True)
for e in EXAMPLES:
    results.append((e, ask(e['say']), ask_clip(synthetic(e['say']))))
    shown.update(table(results))  # the same table, drawn again with this sentence's row
for name, k in (('typed', 1), ('spoken, synthetic voice', 2)):
    seconds = sorted(r[k]['seconds'] for r in results)
    print('%s: %d of %d right; seconds per request min %.2f, median %.2f, max %.2f' % (
        name, sum(right(r[0], r[k]) for r in results), len(results), seconds[0], seconds[len(seconds) // 2], seconds[-1]))

| # | Say | Gemma should call | Typed: Gemma called | Spoken: Gemma heard | Spoken: Gemma called |
|---|---|---|---|---|---|
| 1 | pick up the small red cube and put it in the box | `pick_place(object: small red cube, place: box)` | `pick_place(object: small red cube, place: box)` | pick up the small red cube and put it in the box | `pick_place(object: small red cube, place: box)` |
| 2 | put the red cube in the box | `pick_place(object: red cube, place: box)` | `pick_place(object: red cube, place: box)` | put the red cube in the box | `pick_place(object: red cube, place: box)` |
| 3 | put the blue ball in the box | `pick_place(object: blue ball, place: box)` | `pick_place(object: blue ball, place: box)` | put the blue ball in the box | `pick_place(object: blue ball, place: box)` |
| 4 | grab the large green cylinder and place it in the box | `pick_place(object: large green cylinder, place: box)` | `pick_place(object: large green cylinder, place: box)` | grab the large green cylinder and place it in the box | `pick_place(object: large green cylinder, place: box)` |
| 5 | pick up the yellow cylinder | `pick_place(object: yellow cylinder)` | `pick_place(object: yellow cylinder)` | pick up the yellow cylinder | `pick_place(object: yellow cylinder)` |
| 6 | take the small blue cube to the bin | `pick_place(object: small blue cube, place: bin)` | `pick_place(object: small blue cube, place: bin)` | take the small blue cube to the bin | `pick_place(object: small blue cube, place: bin)` |
| 7 | place the large red ball in the bin | `pick_place(object: large red ball, place: bin)` | `pick_place(object: large red ball, place: bin)` | pick up the large red ball in the bin | `pick_place(object: large red ball, place: bin)` |
| 8 | pick up the ball | `pick_place(object: ball)` | `pick_place(object: ball)` | pick up the ball | `pick_place(object: ball)` |
| 9 | grab the bottle and hand it to me | `pick_place(object: bottle)` | `pick_place(object: bottle)` | grab the bottle and hand it to me | `pick_place(object: bottle)` |
| 10 | learn the stack | `learn_stack()` | `learn_stack()` | learn the stack | `learn_stack()` |
| 11 | rebuild the stack | `rebuild_stack()` | `rebuild_stack()` | rebuild the stack | `rebuild_stack()` |
| 12 | stop robot | `stop()` | `stop()` | stop robot | `stop()` |
| 13 | go home | `home()` | `home()` | go home | `home()` |
| 14 | wave hello | `wave()` | `wave()` | wave hello | `wave()` |
| 15 | fetch the cube | `fetch_block()` | `fetch_block()` | fetch block | `fetch_block()` |
| 16 | show me your best dance moves | `dance()` | `dance()` | Show me your best dance moves | `dance()` |
| 17 | can you copy my arm movements | `gesture_mimic()` | `gesture_mimic()` | Can you copy my arm movements | `gesture_mimic()` |
| 18 | mirror my hand please | `gesture_mimic()` | `gesture_mimic()` | mirror my hand please | `gesture_mimic()` |
| 19 | go grab the block and bring it to me | `fetch_block()` | `fetch_block()` | go grab the block and bring it to me | `fetch_block(object: block)` |
| 20 | hold this pencil for me | `grip()` | `grip(object: pencil)` | pick up this pencil for me | `pick_place(object: pencil)` **WRONG** |
| 21 | release | `release()` | `release()` | release | `release()` |
| 22 | open your hand | `release()` | `release()` | open your hand | `release()` |
| 23 | drop it | `release()` | `release()` | drop it | `release()` |
| 24 | do not rebuild the stack | `no command` | `no command` | rebuild the stack | `rebuild_stack()` **WRONG** |
| 25 | the stack looks nice | `no command` | `no command` | The stack looks nice | `no command` |
| 26 | I learned a lot today | `no command` | `no command` | I learned a lot today | `no command` |
| 27 | who built this stack | `no command` | `no command` | rebuild this stack | `rebuild_stack()` **WRONG** |
| 28 | the red cube is in the box | `no command` | `no command` | The red cube is in the box | `pick_place(object: red cube, place: box)` **WRONG** |
| 29 | is the ball in the box | `no command` | `no command` | Is the ball in the box | `no command` |
| 30 | what's the weather like today | `no command` | `no command` | What's the weather like today | `no command` |

typed: 30 of 30 right; seconds per request min 0.54, median 0.69, max 1.00
spoken, synthetic voice: 26 of 30 right; seconds per request min 0.73, median 0.88, max 1.17


## 6. Your voice, one clip

Run the cell and speak one sentence right away. It records as long as the session's window, lets you play back what the microphone took, and shows what Gemma heard and called. Set `ROW` to a row number of the table to have the answer checked against that row. The microphone is the system's default input; to use another one, set `MIC` in the first cell.

In [ ]:
ROW = None  # a row number of the table to check the answer against, or None
print('Recording %.0f s. Speak now.' % WINDOW_S, flush=True)
recording = sd.rec(int(WINDOW_S * RATE), samplerate=RATE, channels=1, dtype='int16', device=MIC)
sd.wait()  # returns when the recording is complete
wav = _wav(recording.tobytes(), RATE)
display(Audio(wav))
show(ask_clip(wav), EXAMPLES[ROW - 1] if ROW else None)

## 7. The live voice session

Run the cell and speak. This is the branch's own loop (`vla_pipeline/llm/voice.py`) on your microphone: speech detection, then every 2 s in which it hears speech the last 6 s of sound go to Gemma (`hop_s`, `window_s` in `config/pipeline.yaml`). A command is shown once, as `you:` and `->`; the same command is not shown again within 5 s (`cooldown_s`). A sentence in which Gemma finds no command gives no line, because the session drops it.

The cell listens until you interrupt it. If the microphone cannot be opened, it stops with an error.

In [ ]:
server()
mic_cfg = t.load_voice_config()
mic_cfg['audio']['mic_device'] = MIC
session = open_voice(mic_cfg)  # Gemma already runs, so this connects to it and starts nothing
if session.error:
    raise RuntimeError('The voice session is not ready: %s' % session.error)
session.start(lambda text: show(answer(command_from_json(text))))
try:
    if not session.opened.wait(60) or session.error:
        raise RuntimeError('The microphone or the speech detection could not be opened: %s'
                           % (session.error or 'the session did not report within 60 s'))
    print('Listening to your microphone. Speak; interrupt the cell to stop.', flush=True)
    session._thread.join()  # returns only if the microphone thread ends by itself
    raise RuntimeError('The microphone thread ended by itself.')
except KeyboardInterrupt:
    print('Stopped listening.')
finally:
    session.stop()

## 8. Stop Gemma

In [7]:
if globals().get('gemma') is not None:
    gemma.close()
    gemma = None
    print('Gemma stopped.')

Gemma stopped.
